## 📊 Etapa Gold - Consumo de los datos limpios y extructurados.
#### 📁 Paso 1: Análisis del comportamiento de las ventas:

In [0]:
from pyspark.sql.functions import (
    col,
    upper,
    trim,
    abs,
    first,
    regexp_replace,
    expr,
    round,
    countDistinct,
    sum,
    avg
)

# ==========================================
# LEER DATOS DESDE SILVER
# ==========================================
df = spark.table("workspace.silver.movcomercial")

# ==========================================
# NORMALIZAR CLASE
# ==========================================
df = df.withColumn(
    "CLASE",
    upper(trim(col("CLASE")))
)

# ==========================================
# LIMPIAR Y CONVERTIR CANTIDAD
# ==========================================
df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD"), r"\.", "")
)
df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD_TMP"), ",", ".")
)
df = df.withColumn(
    "CANTIDAD_TMP",
    expr("try_cast(CANTIDAD_TMP as double)")
)

# ==========================================
# FILTRAR SOLO VENTAS
# ==========================================
df_ventas = df.filter(
    col("CLASE") == "FV00"
)

# ==========================================
# VENTAS POR PRODUCTO
# ==========================================
ventas_producto = df_ventas.groupBy(
    "PRODUCTO"
).agg(

    first("PRODUCTONO")
    .alias("NOMBRE_PRODUCTO"),
    # CONTAR FACTURAS DE VENTA
    countDistinct("NUMERO")
    .alias("TOTAL_VENTAS"),
    # SUMAR UNIDADES VENDIDAS
    round(
        abs(sum("CANTIDAD_TMP")),
        2
    ).alias("UNIDADES_VENDIDAS"),
    # PRECIO TOTAL DE VENTA
    round(
        sum("PARCIAL"),
        2
    ).alias("PRECIO_VENTA"),
    # PRECIO TOTAL ANTES DE IVA
    round(
        sum("PARCIALANT"),
        2
    ).alias("PRECIO_ANTES_IVA"),
    # RENTABILIDAD %
    round(
        avg("RENTABILIDAD") * 100,
        2
    ).alias("RENTABILIDAD_PORCENTAJE")

).orderBy(
    col("TOTAL_VENTAS").desc()
)

# ==========================================
# MOSTRAR RESULTADO
# ==========================================
display(ventas_producto)

## 📊 Visualización gráfica.
#### 📁 Comportamiento de las Ventas:

In [0]:
from pyspark.sql.functions import (
    col,
    upper,
    trim,
    abs,
    first,
    regexp_replace,
    expr,
    round,
    countDistinct,
    sum,
    avg
)

import matplotlib.pyplot as plt
# ==========================================
# LEER DATOS DESDE SILVER
# ==========================================
df = spark.table("workspace.silver.movcomercial")

# ==========================================
# NORMALIZAR CLASE
# ==========================================
df = df.withColumn(
    "CLASE",
    upper(trim(col("CLASE")))
)

# ==========================================
# LIMPIAR Y CONVERTIR CANTIDAD
# ==========================================
df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD"), r"\.", "")
)

df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD_TMP"), ",", ".")
)
df = df.withColumn(
    "CANTIDAD_TMP",
    expr("try_cast(CANTIDAD_TMP as double)")
)

# ==========================================
# FILTRAR SOLO VENTAS
# ==========================================
df_ventas = df.filter(
    col("CLASE") == "FV00"
)

# ==========================================
# VENTAS POR PRODUCTO
# ==========================================
ventas_producto = df_ventas.groupBy(
    "PRODUCTO"
).agg(

    first("PRODUCTONO")
    .alias("NOMBRE_PRODUCTO"),
    # CONTAR FACTURAS DE VENTA
    countDistinct("NUMERO")
    .alias("TOTAL_VENTAS"),
    # SUMAR UNIDADES VENDIDAS
    round(
        abs(sum("CANTIDAD_TMP")),
        2
    ).alias("UNIDADES_VENDIDAS"),
    # PRECIO TOTAL DE VENTA
    round(
        sum("PARCIAL"),
        2
    ).alias("PRECIO_VENTA"),
    # PRECIO TOTAL ANTES DE IVA
    round(
        sum("PARCIALANT"),
        2
    ).alias("PRECIO_ANTES_IVA"),
    # RENTABILIDAD %
    round(
        avg("RENTABILIDAD") * 100,
        2
    ).alias("RENTABILIDAD_PORCENTAJE")

).orderBy(
    col("TOTAL_VENTAS").desc()
)

# ==========================================
# TOP 20
# ==========================================
top20 = ventas_producto.limit(20)

# ==========================================
# CONVERTIR A PANDAS
# ==========================================
pdf = top20.toPandas()

# ==========================================
# GRAFICAR
# ==========================================
plt.figure(figsize=(14,8))

plt.bar(
    pdf["NOMBRE_PRODUCTO"],
    pdf["TOTAL_VENTAS"]
)

plt.xticks(rotation=90)

plt.xlabel("Producto")
plt.ylabel("Total Ventas")
plt.title("Top 20 Productos con Más Ventas")

plt.tight_layout()

plt.show()

## 📊 Etapa Gold - Consumo de los datos limpios y extructurados.
#### 📁 Paso 1: Análisis del comportamiento de las Devoluciones:

In [0]:
from pyspark.sql.functions import (
    col,
    upper,
    trim,
    abs,
    first,
    regexp_replace,
    expr,
    round,
    countDistinct,
    sum,
    avg
)

# ==========================================
# LEER DATOS DESDE SILVER
# ==========================================
df = spark.table("workspace.silver.movcomercial")

# ==========================================
# NORMALIZAR CLASE
# ==========================================
df = df.withColumn(
    "CLASE",
    upper(trim(col("CLASE")))
)

# ==========================================
# LIMPIAR Y CONVERTIR CANTIDAD
# ==========================================
df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD"), r"\.", "")
)

df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD_TMP"), ",", ".")
)

df = df.withColumn(
    "CANTIDAD_TMP",
    expr("try_cast(CANTIDAD_TMP as double)")
)

# ==========================================
# FILTRAR SOLO DEVOLUCIONES
# ==========================================
df_devoluciones = df.filter(
    col("CLASE") == "DV00"
)

# ==========================================
# DEVOLUCIONES POR PRODUCTO
# ==========================================
devoluciones_producto = df_devoluciones.groupBy(
    "PRODUCTO"
).agg(

    first("PRODUCTONO")
    .alias("NOMBRE_PRODUCTO"),
    # CONTAR DOCUMENTOS DE DEVOLUCIÓN
    countDistinct("NUMERO")
    .alias("TOTAL_DEVOLUCIONES"),
    # SUMAR UNIDADES DEVUELTAS
    round(
        abs(sum("CANTIDAD_TMP")),
        2
    ).alias("UNIDADES_DEVUELTAS"),
    # VALOR TOTAL DEVUELTO
    round(
        abs(sum("PARCIAL")),
        2
    ).alias("PRECIO_VENTA"),
    # VALOR ANTES DE IVA
    round(
        abs(sum("PARCIALANT")),
        2
    ).alias("PRECIO_ANTES_IVA"),
    # RENTABILIDAD %
    round(
        avg("RENTABILIDAD") * 100,
        2
    ).alias("RENTABILIDAD_PORCENTAJE")
).orderBy(
    col("TOTAL_DEVOLUCIONES").desc()
)

# ==========================================
# MOSTRAR RESULTADO
# ==========================================
display(devoluciones_producto)

## 📊 Visualización gráfica.
#### 📁 Comportamiento de las devoluciones:

In [0]:
from pyspark.sql.functions import (
    col,
    upper,
    trim,
    abs,
    first,
    regexp_replace,
    expr,
    round,
    countDistinct,
    sum,
    avg
)
import matplotlib.pyplot as plt
# ==========================================
# LEER DATOS DESDE SILVER
# ==========================================
df = spark.table("workspace.silver.movcomercial")

# ==========================================
# NORMALIZAR CLASE
# ==========================================
df = df.withColumn(
    "CLASE",
    upper(trim(col("CLASE")))
)

# ==========================================
# LIMPIAR Y CONVERTIR CANTIDAD
# ==========================================
df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD"), r"\.", "")
)

df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD_TMP"), ",", ".")
)

df = df.withColumn(
    "CANTIDAD_TMP",
    expr("try_cast(CANTIDAD_TMP as double)")
)

# ==========================================
# FILTRAR SOLO DEVOLUCIONES
# ==========================================
df_devoluciones = df.filter(
    col("CLASE") == "DV00"
)

# ==========================================
# DEVOLUCIONES POR PRODUCTO
# ==========================================
devoluciones_producto = df_devoluciones.groupBy(
    "PRODUCTO"
).agg(

    first("PRODUCTONO")
    .alias("NOMBRE_PRODUCTO"),
    countDistinct("NUMERO")
    .alias("TOTAL_DEVOLUCIONES"),
    round(
        abs(sum("CANTIDAD_TMP")),
        2
    ).alias("UNIDADES_DEVUELTAS"),
    round(
        abs(sum("PARCIAL")),
        2
    ).alias("PRECIO_VENTA"),
    round(
        abs(sum("PARCIALANT")),
        2
    ).alias("PRECIO_ANTES_IVA"),
    round(
        avg("RENTABILIDAD") * 100,
        2
    ).alias("RENTABILIDAD_PORCENTAJE")

).orderBy(
    col("TOTAL_DEVOLUCIONES").desc()
)

# ==========================================
# TOP 20
# ==========================================
top20 = devoluciones_producto.limit(20)

# ==========================================
# CONVERTIR A PANDAS
# ==========================================
pdf = top20.toPandas()

# ==========================================
# GRAFICAR
# ==========================================
plt.figure(figsize=(14,8))

plt.bar(
    pdf["NOMBRE_PRODUCTO"],
    pdf["TOTAL_DEVOLUCIONES"]
)
plt.xticks(rotation=90)
plt.xlabel("Producto")
plt.ylabel("Total Devoluciones")
plt.title("Top 20 Productos con Más Devoluciones")
plt.tight_layout()
plt.show()